# Student model LoRA SFT — Qwen2.5-1.5B-Instruct (Kaggle/Colab T4)

Fills the gap between `src/teacher_labeling/build_student_sft_jsonl.py` and `src/student_predictions/`.

```
teacher_labeling/build_student_sft_jsonl.py
        |
        v
data/synthetic_aux/student_sft_train.jsonl   (chat-format: system/user/assistant JSON target)
        |
        v
*** this notebook ***  -> LoRA adapter
        |
        v
serve adapter with vLLM (OpenAI-compatible /v1/chat/completions)
        |
        v
src/student_predictions/generate_student_predictions.py
        |
        v
src/evaluation/evaluate_triage_predictions.py
```

`student_sft_train.jsonl` already contains finished `messages` triples — system
prompt, user post, assistant target that is the full `schema.json` JSON object
(risk_tier, confidence, evidence_spans, risk_factors, protective_factors,
cssrs_axes, clinical_rationale, plain_language_summary, recommended_next_step,
escalation_required, uncertainty_flags). The teacher's `clinical_rationale`
field already carries the chain-of-thought, so there is **no separate
CoT-rationalization stage here** — this notebook trains directly on the
judged/gated teacher output produced earlier in the pipeline.

### Inputs expected (attach as a Kaggle dataset, or upload to Colab)
- `student_sft_train.jsonl` from `src/teacher_labeling/build_student_sft_jsonl.py`

### Required setup
- Accelerator: GPU T4 (x1 is enough for a 1.5B model in 4-bit).
- Secret `HF_TOKEN` (Qwen2.5-1.5B-Instruct is ungated, but login avoids rate limits).

In [ ]:
# === CELL 1 — install ===
import sys, subprocess, torch
cap = torch.cuda.get_device_capability(0)
print("GPU capability:", cap)
assert cap >= (7, 0), "Use a GPU with bf16 support (e.g. T4)."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets", "huggingface_hub"],
    check=False)
print("install done")

## 2. Paths + Hugging Face auth

In [ ]:
import os, glob
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("HF login OK (secret)")
except Exception:
    try:
        import getpass
        login(getpass.getpass("HF token (leave blank to skip): ") or None)
    except Exception:
        print("skipping HF login")

# --- EDIT if your filename/location differs ---
SFT_JSONL = sorted(glob.glob("/kaggle/input/**/student_sft_train.jsonl", recursive=True)) \
    or sorted(glob.glob("/kaggle/input/**/student_sft*.jsonl", recursive=True)) \
    or sorted(glob.glob("./student_sft_train.jsonl"))
print("sft jsonl:", SFT_JSONL)
assert SFT_JSONL, "Attach student_sft_train.jsonl (output of build_student_sft_jsonl.py) as a dataset."
SFT_JSONL = SFT_JSONL[0]

OUT_DIR = "/kaggle/working/adapter" if os.path.isdir("/kaggle/working") else "./adapter"
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Load the chat-format SFT dataset

Each line already has `{"id":..., "messages": [system, user, assistant], "metadata": {...}}`.
We train directly on `messages` — no relabeling, no rationale synthesis.

In [ ]:
import json
from datasets import Dataset

records = [json.loads(line) for line in open(SFT_JSONL, encoding="utf-8") if line.strip()]
print(f"loaded {len(records)} SFT records")
assert records, "student_sft_train.jsonl is empty — check the gating/judging stages upstream."

ds = Dataset.from_list([{"messages": r["messages"]} for r in records])
test_frac = 0.1 if len(ds) >= 20 else max(1, len(ds) // 10) / max(1, len(ds))
ds = ds.train_test_split(test_size=test_frac, seed=42)
print(ds)
print("\nexample assistant target:\n", ds["train"][0]["messages"][-1]["content"][:500])

## 4. LoRA SFT — Qwen2.5-1.5B-Instruct

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

STUDENT_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

tok = AutoTokenizer.from_pretrained(STUDENT_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16)

peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

cfg = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    max_length=1536,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model, args=cfg, peft_config=peft_cfg,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    processing_class=tok,
)
trainer.train()
trainer.save_model(f"{OUT_DIR}/final")
tok.save_pretrained(f"{OUT_DIR}/final")
print("adapter saved ->", f"{OUT_DIR}/final")

## 5. Sanity check: does the adaptor output schema-valid JSON?

Uses the exact system prompt and `schema.json` from `src/student_predictions/` and
`src/teacher_labeling/` so the check matches what `generate_student_predictions.py`
will send in production.

In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SYSTEM_PROMPT = (
    "You are a research-only crisis triage assistant. "
    "Return only valid JSON matching the triage schema. "
    "Use exact evidence spans copied from the input text. "
    "Do not diagnose, prescribe medication, or provide therapy instructions."
)
REQUIRED_KEYS = [
    "risk_tier", "confidence", "evidence_spans", "risk_factors", "protective_factors",
    "cssrs_axes", "clinical_rationale", "plain_language_summary",
    "recommended_next_step", "escalation_required", "uncertainty_flags",
]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(f"{OUT_DIR}/final")
base = AutoModelForCausalLM.from_pretrained(STUDENT_ID, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16)
m = PeftModel.from_pretrained(base, f"{OUT_DIR}/final").eval()

test_post = ds["test"][0]["messages"][1]["content"]
enc = tok.apply_chat_template(
    [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": test_post}],
    add_generation_prompt=True, return_tensors="pt", return_dict=True).to(m.device)
out = m.generate(**enc, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
raw = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
print(raw)

try:
    parsed = json.loads(raw[raw.find("{"): raw.rfind("}") + 1])
    missing = [k for k in REQUIRED_KEYS if k not in parsed]
    print("\nschema check -> missing keys:", missing or "none")
except Exception as exc:
    print("\nschema check -> FAILED TO PARSE JSON:", exc)

## 6. Hand off to `src/student_predictions/`

Merge the adapter into the base model (or load base+adapter directly) and serve it with
an OpenAI-compatible server, e.g. vLLM:

```bash
pip install vllm
vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --enable-lora \
  --lora-modules local-student-triage-model=/kaggle/working/adapter/final \
  --port 8000
```

Then run the existing prediction pipeline unchanged:

```bash
export LOCAL_LLM_API_KEY=dummy
python src/student_predictions/generate_student_predictions.py \
  --input-jsonl data/student_predictions/test_inputs.jsonl \
  --output-jsonl data/student_predictions/test_predictions.jsonl \
  --model local-student-triage-model \
  --base-url http://localhost:8000/v1 \
  --api-key-env LOCAL_LLM_API_KEY
```

and then `src/evaluation/evaluate_triage_predictions.py` as documented in
`docs/student_model_evaluation_pipeline.md`.

In [ ]:
import os
for root, _, fs in os.walk(f"{OUT_DIR}/final"):
    for f in fs:
        p = os.path.join(root, f)
        print(p, os.path.getsize(p), "bytes")